# Part 4: KV Caching for Frame-Autoregressive Inference

> **Learning objectives:**
> - Understand why KV caching is especially important for video models
> - Implement a VideoKVCache with append, get, and get_and_extend operations
> - Build CachedVideoAttention with finalize and denoise modes
> - Verify correctness against uncached implementation
> - Benchmark the speedup from caching

In [ ]:
import math
import torch
import torch as t
from torch import nn
import torch.nn.functional as F
import numpy as np
import time
import sys
from pathlib import Path

exercises_dir = Path(".").resolve().parent
if str(exercises_dir) not in sys.path:
    sys.path.insert(0, str(exercises_dir))

from part4_far_kv_cache import solutions
from part4_far_kv_cache.tests import (
    test_video_kv_cache, test_cached_video_attention,
    test_cached_causal_dit, test_sample_video_cached,
    test_correctness,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Provided: The Non-Cached CausalDiT

The cells below contain the complete non-cached CausalDiT architecture and `sample_video` function from Part 3. **Just run them** — you don't need to modify anything here. These include the shared building blocks (RMSNorm, GEGLU, RoPE, Patch/UnPatch, numeric encoding, block-causal masking) plus the non-cached `CausalVideoAttention`, `CausalDiTBlock`, and `CausalDiT` classes, and the non-cached `sample_video` sampling loop. Your task in the exercises below is to build cached versions of these components.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Parameter(t.ones(d))

    def forward(self, x):
        return x / ((x ** 2).mean(dim=-1, keepdim=True) + 1e-6).sqrt() * self.w


class GEGLU(nn.Module):
    def __init__(self, d_in, d_mid, d_out):
        super().__init__()
        self.up_proj = nn.Linear(d_in, d_mid, bias=True)
        self.up_gate = nn.Linear(d_in, d_mid, bias=True)
        self.down = nn.Linear(d_mid, d_out, bias=True)
        self.nonlin = nn.SiLU()

    def forward(self, x):
        return self.down(self.up_proj(x) * self.nonlin(self.up_gate(x)))


class NumericEncoding(nn.Module):
    def __init__(self, C=5000, dim=64, n_max=1000):
        super().__init__()
        args = t.exp(-math.log(C) * t.arange(0, dim, 2) / dim)
        args = t.arange(n_max)[:, None] * args[None, :]
        pe = t.empty((n_max, dim))
        pe[:, ::2] = t.sin(args)
        pe[:, 1::2] = t.cos(args)
        self.register_buffer("pe", pe)

    def forward(self, num):
        return self.pe[num]


class RoPE(nn.Module):
    def __init__(self, d_head, n_ctx, C=10000):
        super().__init__()
        thetas = t.exp(-math.log(C) * t.arange(0, d_head, 2) / d_head)
        thetas = thetas.repeat(2, 1).T.flatten()
        positions = t.arange(n_ctx)
        all_thetas = positions.unsqueeze(1) * thetas.unsqueeze(0)
        sins = t.sin(all_thetas)
        coss = t.cos(all_thetas)
        self.register_buffer('sins', sins.unsqueeze(0).unsqueeze(2))
        self.register_buffer('coss', coss.unsqueeze(0).unsqueeze(2))

    def forward(self, x, offset=0):
        """x: (batch, seq, n_head, d_head)"""
        x_perm = t.empty_like(x)
        even = t.arange(0, x.shape[-1], 2, device=x.device)
        odd = t.arange(1, x.shape[-1], 2, device=x.device)
        x_perm[..., even] = -x[..., odd]
        x_perm[..., odd] = x[..., even]
        return self.coss[:, offset:offset + x.shape[1]] * x + self.sins[:, offset:offset + x.shape[1]] * x_perm


class Patch(nn.Module):
    def __init__(self, in_channels=3, out_channels=64, patch_size=2):
        super().__init__()
        self.patch_size = patch_size
        self.out_channels = out_channels
        dim = out_channels
        if dim % 32 == 0 and dim > 32:
            self.init_conv_seq = nn.Sequential(
                nn.Conv2d(in_channels, dim // 2, kernel_size=5, padding=2, stride=1),
                nn.SiLU(),
                nn.GroupNorm(32, dim // 2),
                nn.Conv2d(dim // 2, dim // 2, kernel_size=5, padding=2, stride=1),
                nn.SiLU(),
                nn.GroupNorm(32, dim // 2),
            )
        else:
            self.init_conv_seq = nn.Sequential(
                nn.Conv2d(in_channels, dim // 2, kernel_size=5, padding=2, stride=1),
                nn.SiLU(),
                nn.Conv2d(dim // 2, dim // 2, kernel_size=5, padding=2, stride=1),
                nn.SiLU(),
            )
        self.x_embedder = nn.Linear(patch_size * patch_size * dim // 2, dim, bias=True)

    def patchify(self, x):
        B, C, H, W = x.size()
        ps = self.patch_size
        x = x.view(B, C, H // ps, ps, W // ps, ps)
        x = x.permute(0, 2, 4, 1, 3, 5).flatten(-3).flatten(1, 2)
        return x

    def forward(self, x):
        batch, dur, c, h, w = x.shape
        x = x.reshape(-1, c, h, w)
        x = self.init_conv_seq(x)
        x = self.patchify(x)
        x = self.x_embedder(x)
        x = x.reshape(batch, dur, -1, self.out_channels)
        return x


class UnPatch(nn.Module):
    def __init__(self, height, width, in_channels=64, out_channels=3, patch_size=2):
        super().__init__()
        self.width = width
        self.height = height
        self.patch_size = patch_size
        self.out_channels = out_channels
        self.unpatch = nn.Linear(in_channels, out_channels * patch_size ** 2)

    def forward(self, x):
        batch, dur, seq, d = x.shape
        x = self.unpatch(x)
        x = x.reshape(-1, seq, x.shape[-1])
        c, p = self.out_channels, self.patch_size
        h, w = self.height // p, self.width // p
        x = x.reshape(x.shape[0], h, w, p, p, c)
        x = t.einsum("nhwpqc->nchpwq", x)
        x = x.reshape(x.shape[0], c, h * p, w * p)
        x = x.reshape(batch, dur, c, h * p, w * p)
        return x


def modulate(x, shift, scale):
    """Apply per-frame modulation: cond is (B, T_frames, d), x is (B, S, d)."""
    b, s, d = x.shape
    toks_per_frame = s // shift.shape[1]
    x = x.reshape(b, -1, toks_per_frame, d)
    x = x * (1 + scale[:, :, None, :]) + shift[:, :, None, :]
    x = x.reshape(b, s, d)
    return x


def gate_fn(x, g):
    """Apply per-frame gating: g is (B, T_frames, d), x is (B, S, d)."""
    b, s, d = x.shape
    toks_per_frame = s // g.shape[1]
    x = x.reshape(b, -1, toks_per_frame, d)
    x = x * g[:, :, None, :]
    x = x.reshape(b, s, d)
    return x


def make_block_causal_mask(n_frames, toks_per_frame, device="cpu"):
    """Block-causal mask: token i can attend to j iff frame(i) >= frame(j)."""
    total = n_frames * toks_per_frame
    frame_idx = t.arange(total, device=device) // toks_per_frame
    mask = frame_idx[None, :] > frame_idx[:, None]  # True where BLOCKED
    return mask


# ── Non-cached CausalDiT (for comparison) ─────────────────────────────────

class CausalVideoAttention(nn.Module):
    def __init__(self, d=64, n_head=4, rope=None):
        super().__init__()
        self.n_head = n_head
        self.d = d
        self.d_head = d // n_head
        self.QKV = nn.Linear(d, 3 * d)
        self.O = nn.Linear(d, d)
        self.lnq = RMSNorm(self.d_head)
        self.lnk = RMSNorm(self.d_head)
        self.rope = rope

    def forward(self, x, mask=None):
        b, s, d = x.shape
        q, k, v = self.QKV(x).chunk(3, dim=-1)
        q = q.reshape(b, s, self.n_head, self.d_head)
        k = k.reshape(b, s, self.n_head, self.d_head)
        v = v.reshape(b, s, self.n_head, self.d_head)
        q = self.lnq(q)
        k = self.lnk(k)
        if self.rope is not None:
            q = self.rope(q)
            k = self.rope(k)
        q = q.permute(0, 2, 1, 3)
        k = k.permute(0, 2, 1, 3)
        v = v.permute(0, 2, 1, 3)
        attn = q @ k.transpose(-2, -1)
        if mask is not None:
            attn = attn.masked_fill(mask[None, None, :, :], float('-inf'))
        attn = attn.softmax(dim=-1)
        z = attn @ v
        z = z.permute(0, 2, 1, 3).reshape(b, s, self.d)
        return self.O(z)


class CausalDiTBlock(nn.Module):
    def __init__(self, d=64, n_head=4, exp=4, rope=None):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.selfattn = CausalVideoAttention(d, n_head, rope=rope)
        self.norm2 = RMSNorm(d)
        self.geglu = GEGLU(d, exp * d, d)
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(d, 6 * d, bias=True))

    def forward(self, x, cond, mask=None):
        mu1, sigma1, c1, mu2, sigma2, c2 = self.modulation(cond).chunk(6, dim=-1)
        residual = x
        x = modulate(self.norm1(x), mu1, sigma1)
        x = self.selfattn(x, mask=mask)
        x = residual + gate_fn(x, c1)
        residual = x
        x = modulate(self.norm2(x), mu2, sigma2)
        x = self.geglu(x)
        x = residual + gate_fn(x, c2)
        return x


class CausalDiT(nn.Module):
    """Non-cached version for comparison."""
    def __init__(self, h=24, w=24, n_actions=4, in_channels=3,
                 patch_size=3, n_blocks=8, d=320, n_head=20, exp=4,
                 T=1000, C=5000, n_registers=1, n_window=30):
        super().__init__()
        self.T = T
        self.n_blocks = n_blocks
        self.n_registers = n_registers
        self.n_window = n_window
        self.toks_per_frame = (h // patch_size) * (w // patch_size) + n_registers
        self.patches_per_frame = self.toks_per_frame
        d_head = d // n_head
        rope_ctx = n_window * self.toks_per_frame
        self.rope_seq = RoPE(d_head, rope_ctx, C=C)
        self.blocks = nn.ModuleList(
            [CausalDiTBlock(d, n_head, exp, rope=self.rope_seq) for _ in range(n_blocks)]
        )
        self.patch = Patch(in_channels, d, patch_size)
        self.norm = RMSNorm(d)
        self.unpatch = UnPatch(h, w, d, in_channels, patch_size)
        self.action_emb = nn.Embedding(n_actions, d)
        self.registers = nn.Parameter(t.randn(n_registers, d) * d ** -0.5)
        self.time_emb = NumericEncoding(C=C, dim=d, n_max=T)
        self.time_emb_mixer = nn.Linear(d, d)
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(d, 2 * d, bias=True))

    def forward(self, x, actions, ts):
        B, T_frames, C, H, W = x.shape
        a = self.action_emb(actions)
        ts_scaled = (ts.float() * (self.T - 1)).long()
        cond = self.time_emb_mixer(self.time_emb(ts_scaled)) + a
        z = self.patch(x)  # (B, T_frames, n_patches, d)
        # Append registers to each frame
        regs = self.registers[None, None].expand(B, T_frames, -1, -1)
        zr = t.cat([z, regs], dim=2)  # (B, T_frames, toks_per_frame, d)
        batch, dur, seq, d = zr.shape
        zr = zr.reshape(batch, dur * seq, d)
        mask = make_block_causal_mask(T_frames, self.toks_per_frame, device=x.device)
        for block in self.blocks:
            zr = block(zr, cond, mask=mask)
        mu, sigma = self.modulation(cond).chunk(2, dim=-1)
        zr = modulate(self.norm(zr), mu, sigma)
        zr = zr.reshape(batch, dur, seq, d)
        out = self.unpatch(zr[:, :, :-self.n_registers])
        return out

    @property
    def device(self):
        return self.registers.device

In [ ]:
@t.no_grad()
def sample_video(model, first_frame, actions, n_denoise_steps=30, cfg=0):
    """Non-cached sampling for comparison."""
    B = first_frame.shape[0]
    total_frames = actions.shape[1]
    C, H, W = first_frame.shape[2:]
    video = first_frame.clone()
    for frame_idx in range(1, total_frames):
        z = t.randn(B, 1, C, H, W, device=first_frame.device, dtype=first_frame.dtype)
        denoise_ts = t.linspace(1, 0, n_denoise_steps + 1, device=first_frame.device)
        denoise_ts = 3 * denoise_ts / (2 * denoise_ts + 1)
        for step_idx in range(n_denoise_steps):
            current_video = t.cat([video, z], dim=1)
            n_ctx = current_video.shape[1]
            ts = t.zeros(B, n_ctx, device=first_frame.device)
            ts[:, -1] = denoise_ts[step_idx]
            act = actions[:, :n_ctx]
            v_pred = model(current_video, act, ts)
            if cfg > 0:
                v_uncond = model(current_video, act * 0, ts)
                v_pred = v_uncond + cfg * (v_pred - v_uncond)
            dt = denoise_ts[step_idx] - denoise_ts[step_idx + 1]
            z = z + dt * v_pred[:, -1:]
        video = t.cat([video, z], dim=1)
    return video

## Background: Why Cache for Video?

In Part 3's `sample_video`, generating frame $n$ requires running the *entire* model on all $n$ frames for *each* denoising step. With 30 denoising steps per frame and 16 frames, that's $30 \times 16 = 480$ forward passes, each growing linearly in sequence length.

The key insight: when denoising frame $n$, the representations of frames $1, \ldots, n{-}1$ (which are already clean, with $t=0$) don't change between denoising steps. We can cache their K, V and only compute new K, V for the frame being denoised.

The cache operates in two modes:
- **Finalize**: After a frame is fully denoised, store its K, V permanently in the cache
- **Denoise**: When denoising a new frame, temporarily extend the cached K, V with the current frame's K, V (without modifying the cache)

### RoPE and Sliding Windows

Our model uses **Rotary Position Embeddings (RoPE)** which encode token positions. The cache stores **raw** K, V (before RoPE), and RoPE is **recomputed from position 0** each time attention is computed. This is essential for sliding-window attention: when the cache exceeds the model's context window (`n_window` frames), old frames are evicted, and the remaining frames are re-indexed starting from 0.

### Let's see how slow the non-cached version is

Let's load a pretrained model and generate a few frames with the non-cached `sample_video` to see the problem firsthand.

In [ ]:
import matplotlib.pyplot as plt

# Load pretrained model (downloads from HuggingFace on first run)
pretrained_model, pretrained_config = solutions.load_pretrained_pong(device=device)
print(f"Loaded model: {sum(p.numel() for p in pretrained_model.parameters()):,} parameters")
print(f"Config: {pretrained_config}")

In [ ]:
from matplotlib import animation
from IPython.display import HTML

# Generate 5 frames with the SLOW non-cached method to see the problem
first_frame = t.randn(1, 1, 3, 24, 24, device=device)
actions_demo = t.randint(1, 4, (1, 5), device=device)

if device == "cuda": t.cuda.synchronize()
start = time.time()
with t.autocast(device_type="cuda", dtype=t.bfloat16):
    video_slow = sample_video(pretrained_model, first_frame, actions_demo, n_denoise_steps=6)
if device == "cuda": t.cuda.synchronize()
slow_time = time.time() - start

video_np = (video_slow[0].cpu().permute(0, 2, 3, 1).clamp(-1, 1) * 0.5 + 0.5).numpy()
fig, ax = plt.subplots(figsize=(3.6, 3.6))
im = ax.imshow(video_np[0]); ax.axis("off")
def update(f): im.set_array(video_np[f]); ax.set_title(f"Frame {f}"); return [im]
anim = animation.FuncAnimation(fig, update, frames=len(video_np), interval=167, blit=True)
plt.close()
print(f"Non-cached: {slow_time:.1f}s for {video_slow.shape[1]} frames — let's fix this!")
HTML(anim.to_jshtml())

## Exercise 1 — VideoKVCache
*Difficulty: 🔴🔴🔴⭕⭕ | Importance: 🔵🔵🔵🔵🔵 | ~15 min*

Implement a KV cache that stores **raw** (pre-norm, pre-RoPE) keys and values with three operations:
- `append(layer_idx, k, v)`: permanently add K, V to the cache (evict oldest if over `max_seq_len`)
- `get_cached_kv(layer_idx)`: retrieve cached K, V
- `get_and_extend(layer_idx, new_k, new_v)`: return cached + new K, V *without* modifying the cache

The `max_seq_len` parameter enables sliding-window attention: when the cache would exceed this limit, the oldest tokens are evicted.

In [ ]:
class VideoKVCache:
    """KV Cache for frame-autoregressive video generation.

    Stores raw (pre-norm, pre-RoPE) keys and values so that RoPE can be
    recomputed from position 0 each time — essential for sliding-window
    attention where absolute positions shift as old frames are evicted.

    Args:
        n_layers: Number of transformer layers.
        max_seq_len: Maximum cached sequence length. When exceeded, the
            oldest tokens are evicted (sliding window).
    """
    def __init__(self, n_layers, max_seq_len=None):
        self.cache = [None] * n_layers
        self.max_seq_len = max_seq_len

    def append(self, layer_idx, keys, values):
        """Permanently add keys/values to cache for this layer."""
        if self.cache[layer_idx] is None:
            self.cache[layer_idx] = (keys, values)
        else:
            prev_k, prev_v = self.cache[layer_idx]
            self.cache[layer_idx] = (
                t.cat([prev_k, keys], dim=2),
                t.cat([prev_v, values], dim=2),
            )
        # Sliding window: evict oldest tokens if over capacity
        if self.max_seq_len is not None:
            k, v = self.cache[layer_idx]
            if k.shape[2] > self.max_seq_len:
                excess = k.shape[2] - self.max_seq_len
                self.cache[layer_idx] = (k[:, :, excess:], v[:, :, excess:])

    def get_cached_kv(self, layer_idx):
        """Return (cached_keys, cached_values) or (None, None)."""
        if self.cache[layer_idx] is None:
            return None, None
        return self.cache[layer_idx]

    def get_and_extend(self, layer_idx, new_keys, new_values):
        """Return (cached + new keys, cached + new values) WITHOUT modifying cache."""
        cached_k, cached_v = self.get_cached_kv(layer_idx)
        if cached_k is None:
            return new_keys, new_values
        return (
            t.cat([cached_k, new_keys], dim=2),
            t.cat([cached_v, new_values], dim=2),
        )

    @property
    def cached_seq_len(self):
        """Return the cached sequence length (from layer 0)."""
        if self.cache[0] is None:
            return 0
        return self.cache[0][0].shape[2]

In [ ]:
test_video_kv_cache(VideoKVCache)

## Exercise 2 — CachedVideoAttention
*Difficulty: 🔴🔴🔴🔴⭕ | Importance: 🔵🔵🔵🔵🔵 | ~25 min*

Implement attention with three operating modes:
1. **Standard** (`cache_mode=None`): apply norm + RoPE, then regular attention with mask
2. **Finalize** (`cache_mode="finalize"`): store raw K, V in cache, retrieve full cached K, V, apply norm + RoPE from position 0, compute attention
3. **Denoise** (`cache_mode="denoise"`): extend cached K, V temporarily, apply norm + RoPE from position 0, compute attention

**Key detail**: The cache stores raw (pre-norm, pre-RoPE) K, V. When computing attention in cached modes, RoPE is recomputed on the full K from position 0. The query Q gets RoPE starting at `offset = len(full_k) - len(q)` so its positions are at the end of the sequence.

In [ ]:
class CachedVideoAttention(nn.Module):
    def __init__(self, d=64, n_head=4, rope=None):
        super().__init__()
        self.n_head = n_head
        self.d = d
        self.d_head = d // n_head
        self.QKV = nn.Linear(d, 3 * d)
        self.O = nn.Linear(d, d)
        self.lnq = RMSNorm(self.d_head)
        self.lnk = RMSNorm(self.d_head)
        self.rope = rope

    def forward(self, x, mask=None, kv_cache=None, layer_idx=None, cache_mode=None):
        b, s, d = x.shape
        q, k, v = self.QKV(x).chunk(3, dim=-1)
        q = q.reshape(b, s, self.n_head, self.d_head)
        k = k.reshape(b, s, self.n_head, self.d_head)
        v = v.reshape(b, s, self.n_head, self.d_head)

        if kv_cache is not None and cache_mode is not None:
            # Cache stores RAW (pre-norm, pre-RoPE) k,v so we can
            # recompute RoPE from position 0 each time — needed for
            # sliding-window attention where old frames are evicted.
            k_raw = k.permute(0, 2, 1, 3)   # (b, n_head, s, d_head)
            v_raw = v.permute(0, 2, 1, 3)

            if cache_mode == "finalize":
                kv_cache.append(layer_idx, k_raw, v_raw)
                k_all_raw, v_all = kv_cache.get_cached_kv(layer_idx)
            elif cache_mode == "denoise":
                k_all_raw, v_all = kv_cache.get_and_extend(layer_idx, k_raw, v_raw)

            # Recompute norm + RoPE on full k from position 0
            k_all = k_all_raw.permute(0, 2, 1, 3)  # (b, s_full, n_head, d_head)
            q = self.lnq(q)
            k_all = self.lnk(k_all)
            if self.rope is not None:
                offset = k_all.shape[1] - s  # q sits at the end
                q = self.rope(q, offset=offset)
                k_all = self.rope(k_all)      # from position 0
            q = q.permute(0, 2, 1, 3)
            k_all = k_all.permute(0, 2, 1, 3)
            out = (q @ k_all.transpose(-2, -1)).softmax(dim=-1) @ v_all

        else:
            # Standard (uncached) mode
            q = self.lnq(q)
            k = self.lnk(k)
            if self.rope is not None:
                q = self.rope(q)
                k = self.rope(k)
            q = q.permute(0, 2, 1, 3)
            k = k.permute(0, 2, 1, 3)
            v = v.permute(0, 2, 1, 3)
            attn = q @ k.transpose(-2, -1)
            if mask is not None:
                attn = attn.masked_fill(mask[None, None, :, :], float('-inf'))
            out = attn.softmax(dim=-1) @ v

        out = out.permute(0, 2, 1, 3).reshape(b, s, self.d)
        return self.O(out)

In [ ]:
test_cached_video_attention(CachedVideoAttention, VideoKVCache, make_block_causal_mask)

## Exercise 3 — CachedCausalDiT
*Difficulty: 🔴🔴🔴⭕⭕ | Importance: 🔵🔵🔵🔵🔵 | ~20 min*

Build the cached version of CausalDiT. Thread `kv_cache`, `layer_idx`, and `cache_mode` through blocks. Only create the block-causal mask when in standard mode with multiple frames.

In [ ]:
class CachedCausalDiTBlock(nn.Module):
    def __init__(self, d=64, n_head=4, exp=4, rope=None):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.selfattn = CachedVideoAttention(d, n_head, rope=rope)
        self.norm2 = RMSNorm(d)
        self.geglu = GEGLU(d, exp * d, d)
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(d, 6 * d, bias=True))

    def forward(self, x, cond, mask=None, kv_cache=None, layer_idx=None, cache_mode=None):
        mu1, sigma1, c1, mu2, sigma2, c2 = self.modulation(cond).chunk(6, dim=-1)
        residual = x
        x = modulate(self.norm1(x), mu1, sigma1)
        x = self.selfattn(x, mask=mask, kv_cache=kv_cache,
                          layer_idx=layer_idx, cache_mode=cache_mode)
        x = residual + gate_fn(x, c1)
        residual = x
        x = modulate(self.norm2(x), mu2, sigma2)
        x = self.geglu(x)
        x = residual + gate_fn(x, c2)
        return x


class CachedCausalDiT(nn.Module):
    def __init__(self, h=24, w=24, n_actions=4, in_channels=3,
                 patch_size=3, n_blocks=8, d=320, n_head=20, exp=4,
                 T=1000, C=5000, n_registers=1, n_window=30):
        super().__init__()
        self.T = T
        self.n_blocks = n_blocks
        self.n_registers = n_registers
        self.n_window = n_window
        self.toks_per_frame = (h // patch_size) * (w // patch_size) + n_registers
        self.patches_per_frame = self.toks_per_frame
        d_head = d // n_head
        rope_ctx = n_window * self.toks_per_frame
        self.rope_seq = RoPE(d_head, rope_ctx, C=C)
        self.blocks = nn.ModuleList(
            [CachedCausalDiTBlock(d, n_head, exp, rope=self.rope_seq) for _ in range(n_blocks)]
        )
        self.patch = Patch(in_channels, d, patch_size)
        self.norm = RMSNorm(d)
        self.unpatch = UnPatch(h, w, d, in_channels, patch_size)
        self.action_emb = nn.Embedding(n_actions, d)
        self.registers = nn.Parameter(t.randn(n_registers, d) * d ** -0.5)
        self.time_emb = NumericEncoding(C=C, dim=d, n_max=T)
        self.time_emb_mixer = nn.Linear(d, d)
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(d, 2 * d, bias=True))

    @property
    def max_cache_len(self):
        """Max tokens that can be cached (leave room for one denoise frame)."""
        return (self.n_window - 1) * self.toks_per_frame

    def forward(self, x, actions, ts, kv_cache=None, cache_mode=None):
        B, T_frames, C, H, W = x.shape
        a = self.action_emb(actions)
        ts_scaled = (ts.float() * (self.T - 1)).long()
        cond = self.time_emb_mixer(self.time_emb(ts_scaled)) + a

        z = self.patch(x)  # (B, T_frames, n_patches, d)
        regs = self.registers[None, None].expand(B, T_frames, -1, -1)
        zr = t.cat([z, regs], dim=2)
        batch, dur, seq, d = zr.shape
        zr = zr.reshape(batch, dur * seq, d)

        mask = None
        if cache_mode is None and T_frames > 1:
            mask = make_block_causal_mask(T_frames, self.toks_per_frame, device=x.device)

        for i, block in enumerate(self.blocks):
            zr = block(zr, cond, mask=mask, kv_cache=kv_cache,
                       layer_idx=i, cache_mode=cache_mode)

        mu, sigma = self.modulation(cond).chunk(2, dim=-1)
        zr = modulate(self.norm(zr), mu, sigma)
        zr = zr.reshape(batch, dur, seq, d)
        out = self.unpatch(zr[:, :, :-self.n_registers])
        return out

    @property
    def device(self):
        return self.registers.device

In [ ]:
test_cached_causal_dit(CachedCausalDiT, VideoKVCache)

## Exercise 4 — Cached Video Sampling
*Difficulty: 🔴🔴🔴🔴🔴 | Importance: 🔵🔵🔵🔵🔵 | ~30 min*

Implement video generation using the KV cache. The sampling loop:
1. **Finalize** the first frame (process with `cache_mode="finalize"`)
2. For each subsequent frame:
   a. Denoise from noise using `cache_mode="denoise"` (cache is read-only)
   b. After denoising, **finalize** the clean frame (permanently add to cache)

In [ ]:
@t.no_grad()
def sample_video_cached(model, first_frame, actions, n_denoise_steps=30, cfg=0):
    """Generate video with KV-cached frame-autoregressive sampling."""
    B = first_frame.shape[0]
    total_frames = actions.shape[1]
    C, H, W = first_frame.shape[2:]

    max_len = getattr(model, 'max_cache_len', None)
    cache_cond = VideoKVCache(model.n_blocks, max_seq_len=max_len)
    cache_uncond = VideoKVCache(model.n_blocks, max_seq_len=max_len) if cfg > 0 else None

    video = first_frame.clone()

    # Finalize the first frame
    ts_zero = t.zeros(B, 1, device=first_frame.device)
    act_first = actions[:, :1]
    model(first_frame, act_first, ts_zero, kv_cache=cache_cond, cache_mode="finalize")
    if cfg > 0:
        model(first_frame, act_first * 0, ts_zero, kv_cache=cache_uncond, cache_mode="finalize")

    for frame_idx in range(1, total_frames):
        z = t.randn(B, 1, C, H, W, device=first_frame.device, dtype=first_frame.dtype)

        denoise_ts = t.linspace(1, 0, n_denoise_steps + 1, device=first_frame.device)
        denoise_ts = 3 * denoise_ts / (2 * denoise_ts + 1)

        act_frame = actions[:, frame_idx:frame_idx + 1]

        for step_idx in range(n_denoise_steps):
            ts_frame = denoise_ts[step_idx] * t.ones(B, 1, device=first_frame.device)

            v_pred = model(z, act_frame, ts_frame,
                           kv_cache=cache_cond, cache_mode="denoise")

            if cfg > 0:
                v_uncond = model(z, act_frame * 0, ts_frame,
                                 kv_cache=cache_uncond, cache_mode="denoise")
                v_pred = v_uncond + cfg * (v_pred - v_uncond)

            dt = denoise_ts[step_idx] - denoise_ts[step_idx + 1]
            z = z + dt * v_pred

        # Finalize: store clean frame's K,V in cache
        model(z, act_frame, ts_zero, kv_cache=cache_cond, cache_mode="finalize")
        if cfg > 0:
            model(z, act_frame * 0, ts_zero, kv_cache=cache_uncond, cache_mode="finalize")

        video = t.cat([video, z], dim=1)

    return video

In [ ]:
test_sample_video_cached(sample_video_cached, CachedCausalDiT)

## Exercise 5 — Verify Correctness and Benchmark
*Difficulty: 🔴🔴⭕⭕⭕ | Importance: 🔵🔵🔵🔵⭕ | ~15 min*

Verify that cached and uncached sampling produce identical results (with the same random seed), then benchmark the speedup.

In [ ]:
test_correctness(
    solutions.CausalDiT, CachedCausalDiT,
    solutions.sample_video, sample_video_cached,
)

# Benchmark
from part4_far_kv_cache.solutions import CausalDiT, sample_video
from part4_far_kv_cache.tests import TEST_CONFIG

model_naive = CausalDiT(**TEST_CONFIG).to(device)
model_cached = CachedCausalDiT(**TEST_CONFIG).to(device)
model_cached.load_state_dict(model_naive.state_dict())

first_frame = t.randn(1, 1, 3, 24, 24, device=device)
act = t.ones(1, 4, dtype=t.long, device=device)

if device == "cuda": t.cuda.synchronize()
start = time.time()
sample_video(model_naive, first_frame, act, n_denoise_steps=3)
if device == "cuda": t.cuda.synchronize()
time_naive = time.time() - start

if device == "cuda": t.cuda.synchronize()
start = time.time()
sample_video_cached(model_cached, first_frame, act, n_denoise_steps=3)
if device == "cuda": t.cuda.synchronize()
time_cached = time.time() - start

print(f"Naive: {time_naive:.3f}s | Cached: {time_cached:.3f}s | Speedup: {time_naive/time_cached:.1f}x")

## Bonus: Generate a Long Pong Video

Now that you have a working cached sampler, let's generate 120 frames with different action sequences and see how much faster it is!

In [ ]:
# Control sequence: 30 frames each of init, nothing, up, down
# Actions: 0=init/unconditional, 1=nothing, 2=up, 3=down
actions = t.cat([
    t.full((1, 30), 0, dtype=t.long),
    t.full((1, 30), 1, dtype=t.long),
    t.full((1, 30), 2, dtype=t.long),
    t.full((1, 30), 3, dtype=t.long),
], dim=1).to(device)

first_frame = t.randn(1, 1, 3, 24, 24, device=device)

if device == "cuda": t.cuda.synchronize()
start = time.time()
with t.autocast(device_type="cuda", dtype=t.bfloat16):
    video = sample_video_cached(pretrained_model, first_frame, actions, n_denoise_steps=6, cfg=0)
if device == "cuda": t.cuda.synchronize()
fast_time = time.time() - start

print(f"Generated {video.shape[1]} frames in {fast_time:.1f}s with caching")

video_np = (video[0].cpu().permute(0, 2, 3, 1).clamp(-1, 1) * 0.5 + 0.5).numpy()
fig, ax = plt.subplots(figsize=(3.6, 3.6))
im = ax.imshow(video_np[0]); ax.axis("off")
def update(f): im.set_array(video_np[f]); ax.set_title(f"Frame {f} | action={actions[0,f].item()}"); return [im]
anim = animation.FuncAnimation(fig, update, frames=len(video_np), interval=33, blit=True)
plt.close()
HTML(anim.to_jshtml())

## Summary

You've implemented KV caching for frame-autoregressive video generation:
- **VideoKVCache** stores raw K, V with sliding-window eviction
- **RoPE recomputation** from position 0 enables unbounded generation with a fixed context window
- **Finalize/Denoise modes** separate permanent storage from temporary extension
- Cached sampling avoids redundant computation on already-denoised frames

Congratulations — you've built a complete video world model pipeline from scratch!